# 00. Exploratory Data Analysis - CIFAR-10

Analisis dataset CIFAR-10 sebelum training.

**Contents:**
- Dataset overview
- Class distribution
- Sample visualization
- Image statistics
- Data augmentation preview

In [ ]:
import sys
sys.path.append('..')

import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")

## 1. Load Dataset

In [ ]:
# Load CIFAR-10 without transforms for analysis
train_dataset = torchvision.datasets.CIFAR10(
    root='../data', train=True, download=True, transform=transforms.ToTensor()
)
test_dataset = torchvision.datasets.CIFAR10(
    root='../data', train=False, download=True, transform=transforms.ToTensor()
)

# Class names
classes = ('airplane', 'automobile', 'bird', 'cat', 'deer', 
           'dog', 'frog', 'horse', 'ship', 'truck')

print(f"\n{'='*50}")
print("CIFAR-10 Dataset Overview")
print(f"{'='*50}")
print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")
print(f"Total samples: {len(train_dataset) + len(test_dataset):,}")
print(f"Number of classes: {len(classes)}")
print(f"Image size: 32x32x3 (RGB)")
print(f"Classes: {classes}")

## 2. Class Distribution

In [ ]:
# Count samples per class
train_labels = [train_dataset[i][1] for i in range(len(train_dataset))]
test_labels = [test_dataset[i][1] for i in range(len(test_dataset))]

train_counts = Counter(train_labels)
test_counts = Counter(test_labels)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(classes))
width = 0.7

# Training set
train_values = [train_counts[i] for i in range(len(classes))]
bars1 = axes[0].bar(x, train_values, width, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of Samples')
axes[0].set_title('Training Set Distribution (50,000 samples)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(classes, rotation=45, ha='right')
axes[0].axhline(y=5000, color='red', linestyle='--', label='Perfect balance (5000)')
axes[0].legend()

# Test set
test_values = [test_counts[i] for i in range(len(classes))]
bars2 = axes[1].bar(x, test_values, width, color='coral', edgecolor='black')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Number of Samples')
axes[1].set_title('Test Set Distribution (10,000 samples)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(classes, rotation=45, ha='right')
axes[1].axhline(y=1000, color='red', linestyle='--', label='Perfect balance (1000)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../weights/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ CIFAR-10 is perfectly balanced!")
print(f"   - Each class has exactly 5,000 training & 1,000 test samples")

## 3. Sample Images per Class

In [ ]:
# Get sample images for each class
fig, axes = plt.subplots(10, 10, figsize=(15, 15))

for class_idx in range(10):
    # Get indices for this class
    class_indices = [i for i, (_, label) in enumerate(train_dataset) if label == class_idx][:10]
    
    for i, idx in enumerate(class_indices):
        img, _ = train_dataset[idx]
        img = img.permute(1, 2, 0).numpy()  # CHW -> HWC
        
        axes[class_idx, i].imshow(img)
        axes[class_idx, i].axis('off')
        
        if i == 0:
            axes[class_idx, i].set_ylabel(classes[class_idx], fontsize=12, rotation=0, 
                                          ha='right', va='center')

plt.suptitle('Sample Images from Each Class (10 per class)', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../weights/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Image Statistics

In [ ]:
# Calculate mean and std across dataset
loader = torch.utils.data.DataLoader(train_dataset, batch_size=1000, shuffle=False)

mean = torch.zeros(3)
std = torch.zeros(3)
n_samples = 0

for images, _ in loader:
    batch_samples = images.size(0)
    images = images.view(batch_samples, images.size(1), -1)
    mean += images.mean(2).sum(0)
    std += images.std(2).sum(0)
    n_samples += batch_samples

mean /= n_samples
std /= n_samples

print(f"{'='*50}")
print("Image Statistics (Normalized [0,1])")
print(f"{'='*50}")
print(f"Mean (R, G, B): ({mean[0]:.4f}, {mean[1]:.4f}, {mean[2]:.4f})")
print(f"Std  (R, G, B): ({std[0]:.4f}, {std[1]:.4f}, {std[2]:.4f})")
print(f"\nStandard CIFAR-10 normalization values:")
print(f"Mean: (0.4914, 0.4822, 0.4465)")
print(f"Std:  (0.2470, 0.2435, 0.2616)")

## 5. Pixel Value Distribution

In [ ]:
# Sample random images for histogram
sample_indices = np.random.choice(len(train_dataset), 1000, replace=False)
sample_images = torch.stack([train_dataset[i][0] for i in sample_indices])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['red', 'green', 'blue']
channel_names = ['Red', 'Green', 'Blue']

for i, (ax, color, name) in enumerate(zip(axes, colors, channel_names)):
    channel_data = sample_images[:, i, :, :].flatten().numpy()
    ax.hist(channel_data, bins=50, color=color, alpha=0.7, edgecolor='black')
    ax.set_xlabel('Pixel Value')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{name} Channel Distribution')
    ax.axvline(x=channel_data.mean(), color='black', linestyle='--', 
               label=f'Mean: {channel_data.mean():.3f}')
    ax.legend()

plt.tight_layout()
plt.savefig('../weights/pixel_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Data Augmentation Preview

In [ ]:
from torchvision.transforms import v2 as T

# Define augmentations
augmentations = {
    'Original': transforms.Compose([transforms.ToTensor()]),
    'HorizontalFlip': transforms.Compose([
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor()
    ]),
    'RandomCrop': transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor()
    ]),
    'ColorJitter': transforms.Compose([
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor()
    ]),
    'RandomRotation': transforms.Compose([
        transforms.RandomRotation(15),
        transforms.ToTensor()
    ]),
    'Combined': transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor()
    ])
}

# Load raw dataset for augmentation preview
raw_dataset = torchvision.datasets.CIFAR10(
    root='../data', train=True, download=False, transform=None
)

# Sample images
sample_indices = [100, 2000, 5000, 10000]
fig, axes = plt.subplots(len(sample_indices), len(augmentations), figsize=(15, 10))

for row, idx in enumerate(sample_indices):
    img, label = raw_dataset[idx]
    
    for col, (aug_name, aug_transform) in enumerate(augmentations.items()):
        augmented = aug_transform(img)
        axes[row, col].imshow(augmented.permute(1, 2, 0).numpy())
        axes[row, col].axis('off')
        
        if row == 0:
            axes[row, col].set_title(aug_name, fontsize=11)
        if col == 0:
            axes[row, col].set_ylabel(classes[label], fontsize=11, rotation=0, 
                                      ha='right', va='center')

plt.suptitle('Data Augmentation Preview', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../weights/augmentation_preview.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Class Difficulty Analysis

In [ ]:
# Typical confusion between classes (based on literature)
similar_classes = [
    ('cat', 'dog', 'Similar shape, fur texture'),
    ('automobile', 'truck', 'Both vehicles, similar structure'),
    ('airplane', 'bird', 'Both fly, similar silhouette'),
    ('deer', 'horse', 'Similar body shape'),
]

print(f"{'='*60}")
print("Common Misclassification Pairs (Known from Literature)")
print(f"{'='*60}")
for class1, class2, reason in similar_classes:
    print(f"  • {class1:12} ↔ {class2:12} : {reason}")

print(f"\n{'='*60}")
print("Expected Performance by Class")
print(f"{'='*60}")
expected_difficulty = {
    'airplane': 'Easy - Distinct shape',
    'automobile': 'Medium - Confused with truck', 
    'bird': 'Hard - Varied poses, confused with airplane',
    'cat': 'Hard - Confused with dog',
    'deer': 'Medium - Confused with horse',
    'dog': 'Hard - Confused with cat',
    'frog': 'Easy - Distinct color/shape',
    'horse': 'Medium - Confused with deer',
    'ship': 'Easy - Distinct environment (water)',
    'truck': 'Medium - Confused with automobile'
}

for cls, difficulty in expected_difficulty.items():
    print(f"  • {cls:12}: {difficulty}")

## 8. Summary

In [ ]:
print(f"""
{'='*60}
CIFAR-10 Dataset Summary
{'='*60}

📊 Dataset Size:
   • Training: 50,000 images
   • Test: 10,000 images
   • Total: 60,000 images

🖼️ Image Properties:
   • Size: 32 x 32 pixels
   • Channels: 3 (RGB)
   • Pixel Range: [0, 255] → normalized to [0, 1]

📈 Class Balance:
   • Perfectly balanced (5,000 train / 1,000 test per class)
   • No class imbalance issues

🎨 Normalization Values:
   • Mean: (0.4914, 0.4822, 0.4465)
   • Std:  (0.2470, 0.2435, 0.2616)

🔄 Recommended Augmentations:
   • Random Horizontal Flip
   • Random Crop with padding=4
   • Color Jitter (optional)
   • Cutout / Random Erasing (advanced)

🎯 Baseline Targets:
   • ResNet-18: ~93-95% accuracy
   • ResNet-50: ~95-96% accuracy
   • State-of-art: ~99% (with advanced techniques)

{'='*60}
""")

print("\n✅ EDA Complete! Ready for training.")